In [1]:
import shutil
import os
import json
import random
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset


# =============================================================================
# 0. SETUP (local repo / Kaggle)
# Makes the PatchTST backbone importable so the existing
# `from PatchTST_backbone import PatchTST_backbone` line further down resolves.
# In this repo the backbone, layers, and RevIN all live in the `src/` folder
# next to this notebook (src/PatchTST_backbone.py is already the modified
# version), and the parquet data lives in `data/` next to this notebook.
# =============================================================================
import importlib
import sys
import tempfile

# Notebook directory (Jupyter sets CWD to the notebook's folder).
REPO = os.getcwd()
SRC = os.path.join(REPO, "src")
KAGGLE_INPUT_DATA = "/kaggle/input/datasets/enorabarbiercs/datanew"
KAGGLE_INPUT_BACKBONE = "/kaggle/input/datasets/enorabarbiercs/backbone"
KAGGLE_WORK = "/kaggle/working"

if os.path.isdir(KAGGLE_INPUT_DATA) and os.path.isdir("/kaggle"):
    # Original Kaggle behaviour (kept for portability to Enora's environment).
    shutil.copy(os.path.join(KAGGLE_INPUT_BACKBONE, "PatchTST_backbone.py"),
                os.path.join(KAGGLE_WORK, "PatchTST_backbone.py"))
    for f in ["PatchTST_layers.py", "RevIN.py"]:
        shutil.copy(os.path.join(KAGGLE_INPUT_DATA, f), os.path.join(KAGGLE_WORK, f))
    os.makedirs(os.path.join(KAGGLE_WORK, "data"), exist_ok=True)
    for f in os.listdir(KAGGLE_INPUT_DATA):
        if f.endswith(".parquet") or f.endswith(".json"):
            shutil.copy(os.path.join(KAGGLE_INPUT_DATA, f),
                        os.path.join(KAGGLE_WORK, "data", f))
    os.chdir(KAGGLE_WORK)
    _backbone_dir = KAGGLE_WORK
else:
    # Local repo path: the backbone (already modified), layers, and RevIN are
    # in the notebook's `src/` folder. Stage them into a scratch dir under the
    # name `PatchTST_backbone.py` and prepend that dir to sys.path so the
    # `from PatchTST_backbone import PatchTST_backbone` import resolves. Data is
    # loaded with paths relative to this notebook's directory, so we do not
    # change the working directory here.
    assert os.path.isdir(SRC), f"Expected backbone sources at {SRC}"
    assert os.path.isdir(os.path.join(REPO, "data")), f"Expected {REPO}/data"
    _stage = tempfile.mkdtemp(prefix="seq2seq_v2_backbone_")
    shutil.copy(os.path.join(SRC, "PatchTST_backbone.py"),
                os.path.join(_stage, "PatchTST_backbone.py"))
    for f in ["PatchTST_layers.py", "RevIN.py"]:
        shutil.copy(os.path.join(SRC, f), os.path.join(_stage, f))
    _backbone_dir = _stage

# Make the staged backbone importable.
if _backbone_dir not in sys.path:
    sys.path.insert(0, _backbone_dir)

# Important: clear old imported backbone (in case the notebook is re-run in the
# same kernel after editing this cell).
if "PatchTST_backbone" in sys.modules:
    del sys.modules["PatchTST_backbone"]

import PatchTST_backbone
importlib.reload(PatchTST_backbone)
print("Using backbone:", PatchTST_backbone.__file__)


# =============================================================================
# 1. ENVIRONMENT / CONSTANTS
# =============================================================================

IGNORE_INDEX = -100
RECALL_TOLERANCE_FT = 50.0
EDGE_HALF_WIDTH = 75
EDGE_SIGMA_FT = 20.0

DEFAULT_LOG_COLS = ["GR_CLEAN", "RHOB_CLEAN", "DEEPRES_CLEAN"]

DEFAULT_TOP_COLS = [
    "HORDALAND",
    "ROGALAND",
    "SHETLAND",
    "VIKING",
    "VESTLAND",
    "DUNLIN",
]


def reset_seeds(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# =============================================================================
# 2. DATASET
# =============================================================================

class WellLogDataset(Dataset):
    """
    Returns:
        x:      (n_channels, seq_len)
        y:      (seq_len,) zone labels, IGNORE_INDEX where unknown
        y_edge: (seq_len, n_tops) marker-specific Gaussian top targets
                - value in [0, 1] near known marker top
                - 0.0 away from known marker top inside valid log interval
                - -1.0 ignored marker/depth
    """
    def __init__(
        self,
        X: np.ndarray,
        y: np.ndarray,
        y_edge: np.ndarray,
        depth_jitter: int = 0,
        noise_std: float = 0.0,
        training: bool = False,
    ):
        assert X.ndim == 3, f"Expected X shape (n_wells, n_channels, seq_len), got {X.shape}"
        assert y.ndim == 2, f"Expected y shape (n_wells, seq_len), got {y.shape}"
        assert y_edge.ndim == 3, f"Expected y_edge shape (n_wells, seq_len, n_tops), got {y_edge.shape}"

        self.X = torch.from_numpy(X.astype(np.float32))
        self.y = torch.from_numpy(y.astype(np.int64))
        self.y_edge = torch.from_numpy(y_edge.astype(np.float32))

        self.jitter = int(depth_jitter)
        self.noise_std = float(noise_std)
        self.training = bool(training)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].clone()
        y = self.y[idx].clone()
        y_edge = self.y_edge[idx].clone()

        if self.training:
            if self.noise_std > 0:
                valid = (x != 0).float()
                channel_std = x.std(dim=-1, keepdim=True).clamp(min=1e-6)
                noise = torch.randn_like(x) * channel_std * self.noise_std
                x = x + noise * valid

            if self.jitter > 0:
                shift = random.randint(-self.jitter, self.jitter)
                x = self._shift(x, shift, pad_value=0.0)
                y = self._shift(y, shift, pad_value=IGNORE_INDEX)
                y_edge = self._shift(y_edge.T, shift, pad_value=-1.0).T

        return x, y, y_edge

    @staticmethod
    def _shift(t: torch.Tensor, shift: int, pad_value: float) -> torch.Tensor:
        if shift == 0:
            return t
        t = torch.roll(t, shifts=shift, dims=-1)
        if shift > 0:
            t[..., :shift] = pad_value
        else:
            t[..., shift:] = pad_value
        return t


# =============================================================================
# 3. MODEL HEADS
# =============================================================================

class SegmentationHead(nn.Module):
    """
    Input:  z = (bs, n_vars, feat_dim, patch_num)
    Output: seg_logits = (bs, seq_len, n_classes)

    This is the backup head. It learns geological zones and can be used to
    constrain or diagnose the edge-based top picks.
    """
    def __init__(
        self,
        feat_dim: int,
        n_vars: int,
        patch_num: int,
        seq_len: int,
        n_classes: int,
        dropout: float = 0.1,
        hidden: int = 256,
    ):
        super().__init__()
        in_dim = feat_dim * n_vars
        self.seq_len = seq_len
        self.n_classes = n_classes
        self.patch_num = patch_num

        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
        )
        self.upsample = nn.Upsample(size=seq_len, mode="linear", align_corners=False)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        bs = z.size(0)
        z = z.permute(0, 3, 1, 2).reshape(bs, self.patch_num, -1)
        z = self.mlp(z)
        z = self.upsample(z.permute(0, 2, 1))
        return z.permute(0, 2, 1)


class MultiEdgeHead(nn.Module):
    """
    Primary head.

    Predicts one independent top-probability curve per geological marker:
        edge_logits[:, :, 0] = HORDALAND top score
        edge_logits[:, :, 1] = ROGALAND top score
        ...

    Input:  z = (bs, n_vars, feat_dim, patch_num)
    Output: edge_logits = (bs, seq_len, n_tops)
    """
    def __init__(
        self,
        feat_dim: int,
        n_vars: int,
        patch_num: int,
        seq_len: int,
        n_tops: int,
        dropout: float = 0.1,
        hidden: int = 256,
    ):
        super().__init__()
        in_dim = feat_dim * n_vars
        self.seq_len = seq_len
        self.patch_num = patch_num
        self.n_tops = n_tops

        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, n_tops),
        )
        self.upsample = nn.Upsample(size=seq_len, mode="linear", align_corners=False)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        bs = z.size(0)
        z = z.permute(0, 3, 1, 2).reshape(bs, self.patch_num, -1)
        z = self.mlp(z)
        z = self.upsample(z.permute(0, 2, 1))
        return z.permute(0, 2, 1)


class PatchTSTTopPicker(nn.Module):
    """
    PatchTST encoder with two heads:
        1. MultiEdgeHead: primary marker-specific top localization head.
        2. SegmentationHead: backup zone segmentation head.

    Important:
    Your PatchTST_backbone.py must actually pass rope/alibi flags through.
    In the uploaded backbone version, several internal calls hardcoded rope=False
    and alibi=False. Patch those first if you want rope=True to have an effect.
    """
    def __init__(
        self,
        seq_len: int,
        n_classes: int,
        n_channels: int = 3,
        patch_len: int = 16,
        stride: int = 8,
        d_model: int = 128,
        n_heads: int = 8,
        n_layers: int = 3,
        d_ff: int = 256,
        n_markers: int = 0,
        dropout: float = 0.1,
        head_dropout: float = 0.1,
        pe: str = "zeros",
        learn_pe: bool = True,
        rope: bool = False,
        alibi: bool = False,
        revin: bool = False,
    ):
        super().__init__()
        from PatchTST_backbone import PatchTST_backbone

        patch_num = int((seq_len - patch_len) / stride + 1) + 1
        feat_dim = d_model
        n_tops = n_classes - 1

        self.seq_len = seq_len
        self.n_classes = n_classes
        self.n_channels = n_channels
        self.n_tops = n_tops

        self.backbone = PatchTST_backbone(
            c_in=n_channels,
            context_window=seq_len,
            target_window=seq_len,
            patch_len=patch_len,
            n_markers=n_markers,
            stride=stride,
            d_model=d_model,
            n_heads=n_heads,
            n_layers=n_layers,
            d_ff=d_ff,
            norm="BatchNorm",
            dropout=dropout,
            fc_dropout=dropout,
            head_dropout=head_dropout,
            head_type="flatten",
            individual=False,
            revin=revin,
            padding_patch="end",
            pe=pe,
            learn_pe=learn_pe,
            res_attention=True,
            rope=rope,
            alibi=alibi,
        )
        del self.backbone.head

        self.seg_head = SegmentationHead(
            feat_dim=d_model,
            n_vars=n_channels,
            patch_num=patch_num,
            seq_len=seq_len,
            n_classes=n_classes,
            dropout=head_dropout,
        )

        self.edge_head = MultiEdgeHead(
            feat_dim=d_model,
            n_vars=n_channels,
            patch_num=patch_num,
            seq_len=seq_len,
            n_tops=n_tops,
            dropout=head_dropout,
        )

    def _encode(self, x: torch.Tensor) -> torch.Tensor:
        b = self.backbone

        if b.padding_patch == "end":
            x = b.padding_patch_layer(x)

        x = x.unfold(dimension=-1, size=b.patch_len, step=b.stride)
        x = x.permute(0, 1, 3, 2)

        if b.key_padding_mask == "auto":
            x, kpm = b.key_padding_masking(x)
        else:
            kpm = None

        x = torch.nan_to_num(x, nan=0.0)
        z = b.backbone(x, key_padding_mask_tensor=kpm)
        return z

    def forward(self, x: torch.Tensor):
        z = self._encode(x)
        seg_logits = self.seg_head(z)
        edge_logits = self.edge_head(z)
        return seg_logits, edge_logits


# =============================================================================
# 4. LABEL BUILDING
# =============================================================================

def build_tops_seq(
    depth_arr: np.ndarray,
    top_depths: np.ndarray,
    ignore_index: int = IGNORE_INDEX,
) -> np.ndarray:
    """
    Builds zone labels from ordered top depths.

    Classes:
        0 = before top 0
        1 = top 0 -> top 1
        ...
        6 = after top 5

    Unknown regions are IGNORE_INDEX when required adjacent tops are missing.
    """
    n_tops = len(top_depths)
    labels = np.full(len(depth_arr), ignore_index, dtype=np.int64)

    known = ~np.isnan(top_depths)
    known_idx = np.where(known)[0]

    if len(known_idx) == 0:
        return labels

    first_k = known_idx[0]
    if first_k == 0:
        labels[depth_arr < top_depths[0]] = 0

    last_k = known_idx[-1]
    if last_k == n_tops - 1:
        labels[depth_arr >= top_depths[last_k]] = n_tops

    for j in range(len(known_idx) - 1):
        ka = known_idx[j]
        kb = known_idx[j + 1]
        region = (depth_arr >= top_depths[ka]) & (depth_arr < top_depths[kb])
        if kb == ka + 1:
            labels[region] = ka + 1

    return labels


def build_multiedge_seq(
    depth_grid: np.ndarray,
    top_depths: np.ndarray,
    valid_mask: np.ndarray,
    half_width: int = EDGE_HALF_WIDTH,
    sigma_ft: float = EDGE_SIGMA_FT,
) -> np.ndarray:
    """
    Creates Gaussian target curves for marker-specific top localization.

    For each marker k:
        - ignored if marker is missing or outside valid log interval
        - 0.0 for valid depths away from the marker
        - Gaussian peak centered on the true marker depth
    """
    seq_len = len(depth_grid)
    n_tops = len(top_depths)
    y_edge = np.full((seq_len, n_tops), -1.0, dtype=np.float32)

    if not valid_mask.any():
        return y_edge

    valid_min = depth_grid[valid_mask].min()
    valid_max = depth_grid[valid_mask].max()

    for k, td in enumerate(top_depths):
        if np.isnan(td):
            continue
        if td < valid_min or td > valid_max:
            continue

        y_edge[valid_mask, k] = 0.0
        dist = np.abs(depth_grid - td)
        local = valid_mask & (dist <= half_width)
        y_edge[local, k] = np.exp(-0.5 * ((depth_grid[local] - td) / sigma_ft) ** 2)

    return y_edge.astype(np.float32)


def compute_global_depth_bounds(
    df_tops: pd.DataFrame,
    top_cols: List[str],
    margin: float = 500.0,
    lower_pct: float = 25.0,
    upper_pct: float = 75.0,
) -> Tuple[float, float]:
    # Lithoformer-aligned: hard 2000-14000 ft out-of-range band (cell 41 of
    # NPD_2_Lithoformer_G2_final_copy_carbon.ipynb). This replaces the pct-window
    # so the sequence grid is exactly [2000, 14000]. The margin/lower_pct/upper_pct
    # args are intentionally ignored to keep both call sites (train/valid in
    # run_pipeline and the standalone test-eval cell) on the identical fixed grid.
    return 4000.0, 8000.0   # Colorado-aligned depth gate (see build_colorado_aligned.py)


def standardize_logs_train_valid(
    df_train: pd.DataFrame,
    df_valid: pd.DataFrame,
    log_cols: List[str],
    df_test: Optional[pd.DataFrame] = None,
):
    """
    Standardizes logs using training-set statistics only.
    This often helps transformer stability.
    """
    df_train = df_train.copy()
    df_valid = df_valid.copy()
    df_test = None if df_test is None else df_test.copy()

    stats = {}
    for col in log_cols:
        mu = float(df_train[col].mean(skipna=True))
        sd = float(df_train[col].std(skipna=True))
        if not np.isfinite(sd) or sd < 1e-8:
            sd = 1.0
        stats[col] = (mu, sd)
        df_train[col] = (df_train[col] - mu) / sd
        df_valid[col] = (df_valid[col] - mu) / sd
        if df_test is not None:
            df_test[col] = (df_test[col] - mu) / sd

    return df_train, df_valid, df_test, stats


def prepare_sequences(
    df_log: pd.DataFrame,
    df_tops: pd.DataFrame,
    top_cols: List[str],
    global_min: float,
    global_max: float,
    log_cols: List[str] = DEFAULT_LOG_COLS,
    depth_col: str = "DS_INDEX",
    well_col: str = "well_name",
    srate: float = 4.0,
    pad_value: float = 0.0,
    ignore_index: int = IGNORE_INDEX,
    well_list: Optional[List] = None,
    min_top_fraction: float = 0.65,
    edge_half_width: int = EDGE_HALF_WIDTH,
    edge_sigma_ft: float = EDGE_SIGMA_FT,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[Tuple[np.ndarray, np.ndarray]], List]:

    df_log = df_log.copy()
    df_tops = df_tops.copy()
    df_log[well_col] = df_log[well_col].astype(str)
    df_tops.index = df_tops.index.astype(str)

    wells = well_list if (well_list is not None and len(well_list) > 0) else list(df_tops.index)
    wells = [str(w) for w in wells]

    log_wells = set(df_log[well_col].unique())
    tops_wells = set(df_tops.index)
    min_tops = int(np.ceil(min_top_fraction * len(top_cols)))

    wells = [
        w for w in wells
        if w in log_wells
        and w in tops_wells
        and df_tops.loc[w, top_cols].notna().sum() >= min_tops
    ]

    depth_grid = np.arange(global_min, global_max + srate, srate, dtype=np.float32)
    seq_len = len(depth_grid)
    n_channels = len(log_cols)
    n_tops = len(top_cols)

    X = np.full((len(wells), n_channels, seq_len), pad_value, dtype=np.float32)
    y = np.full((len(wells), seq_len), ignore_index, dtype=np.int64)
    y_edge = np.full((len(wells), seq_len, n_tops), -1.0, dtype=np.float32)
    depth_axes = []

    for i, well in enumerate(wells):
        wdf = df_log[df_log[well_col] == well].sort_values(depth_col)
        depth_raw = wdf[depth_col].values.astype(np.float32)

        if len(depth_raw) == 0:
            depth_axes.append((np.array([], dtype=np.float32), np.array([], dtype=np.int64)))
            continue

        valid_mask = (depth_grid >= depth_raw.min()) & (depth_grid <= depth_raw.max())
        valid_idx = np.where(valid_mask)[0]

        for c, col in enumerate(log_cols):
            values = wdf[col].values.astype(np.float32)
            not_nan = ~np.isnan(values)
            if not_nan.sum() > 1 and valid_mask.any():
                X[i, c, valid_mask] = np.interp(
                    depth_grid[valid_mask],
                    depth_raw[not_nan],
                    values[not_nan],
                ).astype(np.float32)

        top_depths = df_tops.loc[well, top_cols].values.astype(float)

        labels_full = build_tops_seq(depth_grid, top_depths, ignore_index=ignore_index)
        labels = np.full(seq_len, ignore_index, dtype=np.int64)
        valid_and_labelled = valid_mask & (labels_full != ignore_index)
        labels[valid_and_labelled] = labels_full[valid_and_labelled]

        y[i] = labels
        y_edge[i] = build_multiedge_seq(
            depth_grid=depth_grid,
            top_depths=top_depths,
            valid_mask=valid_mask,
            half_width=edge_half_width,
            sigma_ft=edge_sigma_ft,
        )

        depth_axes.append((depth_grid[valid_idx].copy(), valid_idx.copy()))

    print(
        f"Prepared {len(wells)} wells | seq_len={seq_len} | "
        f"n_channels={n_channels} | n_classes={len(top_cols) + 1} | "
        f"global range=[{global_min:.1f}, {global_max:.1f}]"
    )

    return X, y, y_edge, depth_axes, wells


# =============================================================================
# 5. LOSSES
# =============================================================================

def effective_number_weights(
    y: np.ndarray,
    n_classes: int,
    beta: float = 0.9999,
    ignore_index: int = IGNORE_INDEX,
    device: torch.device = torch.device("cpu"),
) -> torch.Tensor:
    valid = y[y != ignore_index].flatten()
    class_counts = np.bincount(valid, minlength=n_classes).astype(np.float64)
    class_counts = np.where(class_counts == 0, 1, class_counts)

    effective = 1.0 - np.power(beta, class_counts)
    weights = (1.0 - beta) / effective
    weights = weights / weights.mean()

    print(f"Class counts: {dict(enumerate(class_counts.astype(int)))}")
    print(f"Class weights: {dict(enumerate(np.round(weights, 4)))}")

    return torch.tensor(weights.astype(np.float32), dtype=torch.float32, device=device)


def top_location_loss(
    edge_logits: torch.Tensor,
    y_edge: torch.Tensor,
) -> torch.Tensor:
    """
    Primary loss.

    For each known marker in each well, it applies a softmax over depth.
    This forces the model to answer: WHERE is marker k?

    This is better aligned with top picking than independent BCE at each depth.
    """
    losses = []
    bs, seq_len, n_tops = edge_logits.shape

    for k in range(n_tops):
        logits_k = edge_logits[:, :, k]
        target_k = y_edge[:, :, k]

        valid_mask = target_k >= 0
        positive_mass = target_k.clamp(min=0).sum(dim=1)
        has_marker = positive_mass > 0

        if not has_marker.any():
            continue

        logits_k = logits_k[has_marker]
        target_k = target_k[has_marker].clamp(min=0.0)
        valid_k = valid_mask[has_marker]

        logits_k = logits_k.masked_fill(~valid_k, -1e9)
        target_k = target_k * valid_k.float()

        target_dist = target_k / target_k.sum(dim=1, keepdim=True).clamp(min=1e-8)
        log_probs = torch.log_softmax(logits_k, dim=1)
        loss_k = -(target_dist * log_probs).sum(dim=1).mean()
        losses.append(loss_k)

    if len(losses) == 0:
        return edge_logits.sum() * 0.0

    return torch.stack(losses).mean()


def segmentation_loss(
    seg_logits: torch.Tensor,
    y: torch.Tensor,
    class_weights: Optional[torch.Tensor],
    ignore_index: int = IGNORE_INDEX,
) -> torch.Tensor:
    criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=ignore_index)
    return criterion(seg_logits.permute(0, 2, 1), y)


# =============================================================================
# 6. DECODERS / TOP EXTRACTION
# =============================================================================

def monotonic_viterbi(probs: np.ndarray) -> np.ndarray:
    T, K = probs.shape
    log_probs = np.log(probs + 1e-12)

    dp = np.full((T, K), -np.inf, dtype=np.float64)
    bp = np.zeros((T, K), dtype=np.int32)
    dp[0] = log_probs[0]

    for t in range(1, T):
        for k in range(K):
            candidates = [(dp[t - 1, k], k)]
            if k > 0:
                candidates.append((dp[t - 1, k - 1], k - 1))
            best_score, best_prev = max(candidates, key=lambda x: x[0])
            dp[t, k] = best_score + log_probs[t, k]
            bp[t, k] = best_prev

    path = np.zeros(T, dtype=np.int64)
    path[-1] = int(np.argmax(dp[-1]))
    for t in range(T - 2, -1, -1):
        path[t] = bp[t + 1, path[t + 1]]
    return path


def top_depths_from_step_function(
    depth_arr: np.ndarray,
    zone_preds: np.ndarray,
    n_tops: int,
) -> np.ndarray:
    est = []
    for k in range(1, n_tops + 1):
        idx = np.where((zone_preds[:-1] < k) & (zone_preds[1:] >= k))[0]
        est.append(float(depth_arr[idx[0]]) if len(idx) else np.nan)
    return np.array(est)


def edge_primary_decoder(
    depth_arr: np.ndarray,
    edge_logits_or_probs: np.ndarray,
    top_cols: List[str],
    min_gap_ft: float = 20.0,
    use_softmax: bool = True,
) -> np.ndarray:
    """
    Primary decoder.

    It predicts each top directly from its own marker-specific edge curve.
    A simple min_gap_ft enforces geological ordering.
    """
    n_tops = len(top_cols)
    pred = np.full(n_tops, np.nan, dtype=float)

    if use_softmax:
        scores_all = edge_logits_or_probs.copy()
    else:
        scores_all = np.log(edge_logits_or_probs + 1e-12)

    prev_depth = -np.inf
    for k in range(n_tops):
        scores = scores_all[:, k].copy()
        allowed = depth_arr >= prev_depth + min_gap_ft
        if not allowed.any():
            continue
        scores[~allowed] = -np.inf
        idx = int(np.argmax(scores))
        if np.isfinite(scores[idx]):
            pred[k] = float(depth_arr[idx])
            prev_depth = pred[k]

    return pred


def edge_windowed_decoder(
    depth_arr: np.ndarray,
    edge_logits_or_probs: np.ndarray,
    zone_preds: np.ndarray,
    top_cols: List[str],
    window_ft: float = 500.0,
    min_gap_ft: float = 20.0,
) -> np.ndarray:
    """
    Edge-primary with segmentation backup.

    For each marker:
        1. Get a rough boundary from segmentation/Viterbi.
        2. Search the marker-specific edge maximum around that boundary.
        3. If no rough boundary exists, fall back to global edge argmax.

    This is often more stable than pure global edge argmax.
    """
    n_tops = len(top_cols)
    rough = top_depths_from_step_function(depth_arr, zone_preds, n_tops)
    pred = np.full(n_tops, np.nan, dtype=float)
    prev_depth = -np.inf

    for k in range(n_tops):
        scores = edge_logits_or_probs[:, k].copy()
        allowed = depth_arr >= prev_depth + min_gap_ft

        if not np.isnan(rough[k]):
            allowed = allowed & (np.abs(depth_arr - rough[k]) <= window_ft)

        if not allowed.any():
            allowed = depth_arr >= prev_depth + min_gap_ft

        if not allowed.any():
            continue

        scores[~allowed] = -np.inf
        idx = int(np.argmax(scores))
        if np.isfinite(scores[idx]):
            pred[k] = float(depth_arr[idx])
            prev_depth = pred[k]

    return pred


def predicted_tops_from_result(
    df: pd.DataFrame,
    top_cols: List[str],
    mode: str = "edge_primary",
    window_ft: float = 500.0,
    min_gap_ft: float = 20.0,
) -> np.ndarray:
    depth_arr = df["DEPTH"].values.astype(float)
    zone_preds = df["predicted_zone"].values.astype(int)
    edge_logits = np.stack([df[f"edge_logit_{c}"].values for c in top_cols], axis=1)

    if mode == "edge_primary":
        return edge_primary_decoder(depth_arr, edge_logits, top_cols, min_gap_ft=min_gap_ft, use_softmax=True)

    if mode == "edge_windowed":
        return edge_windowed_decoder(depth_arr, edge_logits, zone_preds, top_cols, window_ft=window_ft, min_gap_ft=min_gap_ft)

    if mode == "viterbi":
        return top_depths_from_step_function(depth_arr, zone_preds, len(top_cols))

    raise ValueError(f"Unknown top extraction mode: {mode}")


# =============================================================================
# 7. TRAINING / EVALUATION
# =============================================================================

def train_one_epoch(
    model: PatchTSTTopPicker,
    loader: DataLoader,
    optimiser: torch.optim.Optimizer,
    device: torch.device,
    class_weights: Optional[torch.Tensor] = None,
    ignore_index: int = IGNORE_INDEX,
    top_weight: float = 1.0,
    seg_weight: float = 0.25,
) -> Dict[str, float]:
    model.train()
    total_loss = 0.0
    total_top = 0.0
    total_seg = 0.0
    n = 0

    for X_batch, y_batch, y_edge_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        y_edge_batch = y_edge_batch.to(device)

        optimiser.zero_grad()
        seg_logits, edge_logits = model(X_batch)

        loss_top = top_location_loss(edge_logits, y_edge_batch)
        loss_seg = segmentation_loss(seg_logits, y_batch, class_weights, ignore_index=ignore_index)
        loss = top_weight * loss_top + seg_weight * loss_seg

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()

        bs = len(y_batch)
        total_loss += float(loss.item()) * bs
        total_top += float(loss_top.item()) * bs
        total_seg += float(loss_seg.item()) * bs
        n += bs

    return {
        "loss": total_loss / max(n, 1),
        "top_loss": total_top / max(n, 1),
        "seg_loss": total_seg / max(n, 1),
    }


@torch.no_grad()
def evaluate_loss_acc(
    model: PatchTSTTopPicker,
    loader: DataLoader,
    device: torch.device,
    ignore_index: int = IGNORE_INDEX,
) -> Dict[str, float]:
    model.eval()
    total_top = 0.0
    total_seg = 0.0
    correct = 0
    total = 0
    n = 0

    for X_batch, y_batch, y_edge_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        y_edge_batch = y_edge_batch.to(device)

        seg_logits, edge_logits = model(X_batch)
        loss_top = top_location_loss(edge_logits, y_edge_batch)
        loss_seg = segmentation_loss(seg_logits, y_batch, class_weights=None, ignore_index=ignore_index)

        preds = seg_logits.argmax(dim=-1)
        valid = y_batch != ignore_index
        correct += ((preds == y_batch) & valid).sum().item()
        total += valid.sum().item()

        bs = len(y_batch)
        total_top += float(loss_top.item()) * bs
        total_seg += float(loss_seg.item()) * bs
        n += bs

    return {
        "top_loss": total_top / max(n, 1),
        "seg_loss": total_seg / max(n, 1),
        "seg_acc": correct / total if total > 0 else np.nan,
    }


@torch.no_grad()
def predict(
    model: PatchTSTTopPicker,
    X: np.ndarray,
    depth_axes: List,
    log_cols: List[str] = DEFAULT_LOG_COLS,
    top_cols: List[str] = DEFAULT_TOP_COLS,
    batch_size: int = 16,
    zone_decoder: str = "viterbi",
) -> List[pd.DataFrame]:
    device = next(model.parameters()).device
    model.eval()
    results = []

    for start in range(0, len(X), batch_size):
        batch = torch.from_numpy(X[start:start + batch_size]).to(device)
        seg_logits, edge_logits = model(batch)

        seg_probs = torch.softmax(seg_logits, dim=-1).cpu().numpy()
        edge_logits_np = edge_logits.cpu().numpy()
        edge_probs_np = torch.softmax(edge_logits, dim=1).cpu().numpy()
        n_classes = seg_logits.shape[-1]

        for i in range(batch.shape[0]):
            well_idx = start + i
            depth_arr, valid_idx = depth_axes[well_idx]

            if len(valid_idx) == 0:
                continue

            probs_valid = seg_probs[i, valid_idx]
            edge_logits_valid = edge_logits_np[i, valid_idx]
            edge_probs_valid = edge_probs_np[i, valid_idx]

            if zone_decoder == "argmax":
                zone_preds = probs_valid.argmax(axis=-1)
            elif zone_decoder == "viterbi":
                zone_preds = monotonic_viterbi(probs_valid)
            else:
                raise ValueError(f"Unknown zone_decoder: {zone_decoder}")

            row_dict = {"DEPTH": depth_arr}

            for c, col in enumerate(log_cols):
                row_dict[col] = X[well_idx, c, valid_idx]

            row_dict["predicted_zone"] = zone_preds

            for k in range(n_classes):
                row_dict[f"prob_zone_{k}"] = probs_valid[:, k]

            for k, col in enumerate(top_cols):
                row_dict[f"edge_logit_{col}"] = edge_logits_valid[:, k]
                row_dict[f"edge_prob_{col}"] = edge_probs_valid[:, k]

            row_dict["edge_prob_max"] = edge_probs_valid.max(axis=1)
            results.append(pd.DataFrame(row_dict))

    return results


def summarize_eval_df(
    df_eval: pd.DataFrame,
    top_cols: List[str],
    tolerance: float = RECALL_TOLERANCE_FT,
):
    rows = []
    sum_tp, sum_n = 0, 0
    recalls = []

    for col in top_cols:
        gt_col = f"{col}_gt"
        err_col = f"{col}_err"
        valid = df_eval[err_col].dropna()
        gt_v = df_eval[gt_col].values.astype(float)
        err_v = df_eval[err_col].values.astype(float)

        n_ann = int((~np.isnan(gt_v)).sum())
        tp = int((~np.isnan(err_v) & (err_v <= tolerance)).sum())
        rec = tp / n_ann if n_ann > 0 else np.nan

        rows.append({
            "top": col,
            "MAE": float(valid.mean()) if len(valid) else np.nan,
            "Median": float(valid.median()) if len(valid) else np.nan,
            f"Recall@{int(tolerance)}ft": rec,
            "TP": tp,
            "n": n_ann,
        })

        if np.isfinite(rec):
            recalls.append(rec)
        sum_tp += tp
        sum_n += n_ann

    summary = pd.DataFrame(rows)
    overall = {
        "macro_recall_at_tol": float(np.mean(recalls)) if recalls else np.nan,
        "pooled_recall_at_tol": float(sum_tp / sum_n) if sum_n > 0 else np.nan,
        "tolerance_ft": tolerance,
    }
    return summary, overall


def evaluate_top_picking(
    results: List[pd.DataFrame],
    well_names: List,
    df_tops: pd.DataFrame,
    top_cols: List[str],
    tolerance: float = RECALL_TOLERANCE_FT,
    mode: str = "edge_primary",
    window_ft: float = 500.0,
    min_gap_ft: float = 20.0,
    verbose: bool = True,
) -> pd.DataFrame:
    rows = []
    df_tops = df_tops.copy()
    df_tops.index = df_tops.index.astype(str)

    for well, df in zip(well_names, results):
        well = str(well)
        if well not in df_tops.index:
            continue

        est = predicted_tops_from_result(
            df,
            top_cols=top_cols,
            mode=mode,
            window_ft=window_ft,
            min_gap_ft=min_gap_ft,
        )
        gt = df_tops.loc[well, top_cols].values.astype(float)

        row = {"well": well}
        for i, col in enumerate(top_cols):
            row[f"{col}_gt"] = gt[i]
            row[f"{col}_pred"] = est[i]
            row[f"{col}_err"] = abs(est[i] - gt[i]) if not (np.isnan(gt[i]) or np.isnan(est[i])) else np.nan
        rows.append(row)

    df_eval = pd.DataFrame(rows)
    summary, overall = summarize_eval_df(df_eval, top_cols, tolerance=tolerance)

    if verbose:
        tol_int = int(tolerance)
        print(f"\nTop-picking errors and recall@{tol_int}ft | mode={mode}")
        print(summary.to_string(index=False))
        print(f"\nMacro-recall@{tol_int}ft (PRIMARY): {overall['macro_recall_at_tol']:.4f}")
        print(f"Pooled-recall@{tol_int}ft         : {overall['pooled_recall_at_tol']:.4f}")

    return df_eval


def evaluate_segmentation_metrics(
    results: List[pd.DataFrame],
    well_names: List,
    df_tops: pd.DataFrame,
    top_cols: List[str],
):
    all_preds, all_true = [], []
    df_tops = df_tops.copy()
    df_tops.index = df_tops.index.astype(str)

    for well, df in zip(well_names, results):
        well = str(well)
        if well not in df_tops.index:
            continue

        top_depths = df_tops.loc[well, top_cols].values.astype(float)
        true_labels = build_tops_seq(df["DEPTH"].values, top_depths, ignore_index=IGNORE_INDEX)
        pred_labels = df["predicted_zone"].values

        known = true_labels != IGNORE_INDEX
        all_true.extend(true_labels[known].tolist())
        all_preds.extend(pred_labels[known].tolist())

    all_true = np.array(all_true)
    all_preds = np.array(all_preds)

    n_classes = len(top_cols) + 1
    zone_names = [f"zone_{i}" for i in range(n_classes)]
    present = sorted(set(all_true) | set(all_preds))
    present_names = [zone_names[c] for c in present if c < n_classes]

    accuracy = (all_true == all_preds).mean() if len(all_true) else np.nan

    per_class_recall = {}
    for c in present:
        mask_c = all_true == c
        per_class_recall[c] = float((all_preds[mask_c] == c).mean()) if mask_c.sum() > 0 else np.nan
    macro_recall = float(np.nanmean(list(per_class_recall.values()))) if per_class_recall else np.nan

    print(f"\nOverall depth-sample accuracy : {accuracy:.4f}  (on {len(all_true):,} labelled points)")
    print(f"Macro-average recall          : {macro_recall:.4f}")
    recall_str = "  ".join(f"{zone_names[c]}={per_class_recall[c]:.3f}" for c in present if c < n_classes)
    print(f"Per-class recall              : {recall_str}")

    print("\nPer-class metrics:")
    print(classification_report(
        all_true,
        all_preds,
        labels=present,
        target_names=present_names,
        zero_division=0,
    ))

    cm = confusion_matrix(all_true, all_preds, labels=list(range(n_classes)))
    print("Confusion matrix (rows=true, cols=pred):")
    print(pd.DataFrame(cm, index=zone_names, columns=zone_names))

    return accuracy


def decoder_diagnostic(
    model: PatchTSTTopPicker,
    X: np.ndarray,
    depth_axes: List,
    well_names: List,
    df_tops: pd.DataFrame,
    top_cols: List[str],
    log_cols: List[str] = DEFAULT_LOG_COLS,
    batch_size: int = 16,
    windows: Optional[List[float]] = None,
) -> pd.DataFrame:
    if windows is None:
        windows = [100.0, 200.0, 300.0, 500.0, 800.0]

    rows = []
    results = predict(
        model,
        X,
        depth_axes,
        log_cols=log_cols,
        top_cols=top_cols,
        batch_size=batch_size,
        zone_decoder="viterbi",
    )

    for mode in ["edge_primary", "viterbi"]:
        df_eval = evaluate_top_picking(
            results,
            well_names,
            df_tops,
            top_cols,
            mode=mode,
            verbose=False,
        )
        _, overall = summarize_eval_df(df_eval, top_cols)
        rows.append({"decoder": mode, "param": 0.0, **overall})

    for w in windows:
        df_eval = evaluate_top_picking(
            results,
            well_names,
            df_tops,
            top_cols,
            mode="edge_windowed",
            window_ft=w,
            verbose=False,
        )
        _, overall = summarize_eval_df(df_eval, top_cols)
        rows.append({"decoder": "edge_windowed", "param": w, **overall})

    diag = pd.DataFrame(rows).sort_values("macro_recall_at_tol", ascending=False)
    print("\nDecoder diagnostic summary:")
    print(diag.to_string(index=False))
    return diag


# =============================================================================
# 8. TRAIN MODEL
# =============================================================================

def train_model(
    model: PatchTSTTopPicker,
    X_train: np.ndarray,
    y_train: np.ndarray,
    y_edge_train: np.ndarray,
    X_valid: np.ndarray,
    y_valid: np.ndarray,
    y_edge_valid: np.ndarray,
    depth_axes_valid: List,
    valid_wells: List,
    df_tops_valid: pd.DataFrame,
    top_cols: List[str],
    log_cols: List[str] = DEFAULT_LOG_COLS,
    epochs: int = 50,
    batch_size: int = 16,
    lr: float = 1e-4,
    weight_decay: float = 1e-4,
    beta: float = 0.99999,
    depth_jitter: int = 0,
    noise_std: float = 0.05,
    top_weight: float = 1.0,
    seg_weight: float = 0.25,
    selection_mode: str = "edge_primary",
    save_path: str = "patchtst_edge_primary.pt",
    device_str: str = "auto",
    ignore_index: int = IGNORE_INDEX,
) -> PatchTSTTopPicker:

    device = (
        torch.device("cuda" if torch.cuda.is_available() else "cpu")
        if device_str == "auto"
        else torch.device(device_str)
    )
    print(f"Training on {device}")

    model = model.to(device)
    n_classes = model.seg_head.n_classes

    class_weights = effective_number_weights(
        y_train,
        n_classes,
        beta=beta,
        ignore_index=ignore_index,
        device=device,
    )

    train_dl = DataLoader(
        WellLogDataset(
            X_train,
            y_train,
            y_edge_train,
            depth_jitter=depth_jitter,
            noise_std=noise_std,
            training=True,
        ),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )

    valid_dl = DataLoader(
        WellLogDataset(X_valid, y_valid, y_edge_valid, training=False),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {n_params:,}")

    optimiser = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=epochs)

    best_recall = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        train_metrics = train_one_epoch(
            model=model,
            loader=train_dl,
            optimiser=optimiser,
            device=device,
            class_weights=class_weights,
            ignore_index=ignore_index,
            top_weight=top_weight,
            seg_weight=seg_weight,
        )

        val_metrics = evaluate_loss_acc(model, valid_dl, device, ignore_index=ignore_index)
        scheduler.step()

        val_results = predict(
            model,
            X_valid,
            depth_axes_valid,
            log_cols=log_cols,
            top_cols=top_cols,
            batch_size=batch_size,
            zone_decoder="viterbi",
        )

        df_eval_valid = evaluate_top_picking(
            val_results,
            valid_wells,
            df_tops_valid,
            top_cols,
            tolerance=RECALL_TOLERANCE_FT,
            mode=selection_mode,
            verbose=False,
        )
        _, overall = summarize_eval_df(df_eval_valid, top_cols, tolerance=RECALL_TOLERANCE_FT)
        val_recall_at_50 = overall["macro_recall_at_tol"]

        print(
            f"Epoch {epoch:3d}/{epochs} | "
            f"train_loss={train_metrics['loss']:.4f} | "
            f"train_top={train_metrics['top_loss']:.4f} | "
            f"train_seg={train_metrics['seg_loss']:.4f} | "
            f"val_top={val_metrics['top_loss']:.4f} | "
            f"val_seg={val_metrics['seg_loss']:.4f} | "
            f"val_seg_acc={val_metrics['seg_acc']:.4f} | "
            f"val_macro_recall@50={val_recall_at_50:.4f} | "
            f"mode={selection_mode}"
        )

        if np.isfinite(val_recall_at_50) and val_recall_at_50 > best_recall:
            best_recall = val_recall_at_50
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            torch.save(best_state, save_path)
            print(f"  -> Saved best checkpoint: {save_path} | val_macro_recall@50={best_recall:.4f}")

    if best_state is None:
        raise RuntimeError("Training produced no valid checkpoint.")

    model.load_state_dict(best_state)
    model.to(device)
    return model


# =============================================================================
# 9. VISUALIZATION
# =============================================================================

def plot_prediction(
    df_result: pd.DataFrame,
    log_cols: List[str] = DEFAULT_LOG_COLS,
    df_tops: Optional[pd.DataFrame] = None,
    top_cols: Optional[List[str]] = None,
    well_name: str = "",
    save_fig: Optional[str] = None,
    show_probs: bool = True,
):
    try:
        import matplotlib.pyplot as plt
        import matplotlib.ticker as ticker
    except ImportError:
        print("matplotlib not installed; skipping plot.")
        return

    depth = df_result["DEPTH"].values
    zone = df_result["predicted_zone"].values

    prob_cols = [c for c in df_result.columns if c.startswith("prob_zone_")]
    edge_cols = [c for c in df_result.columns if c.startswith("edge_prob_") and c != "edge_prob_max"]

    n_logs = len([c for c in log_cols if c in df_result.columns])
    n_prob_axes = 1 if (show_probs and prob_cols) else 0
    n_edge_axes = 1 if (show_probs and edge_cols) else 0
    n_axes = n_logs + 1 + n_prob_axes + n_edge_axes

    fig, axes = plt.subplots(1, n_axes, figsize=(3 * n_axes, 10), sharey=True)
    if n_axes == 1:
        axes = [axes]

    colors = plt.cm.tab10.colors
    ax_idx = 0

    for col in log_cols:
        if col not in df_result.columns:
            continue
        ax = axes[ax_idx]
        ax.plot(df_result[col].values, depth, lw=0.8)
        ax.set_xlabel(col)
        ax.set_title(col)
        if ax_idx == 0:
            ax.set_ylabel("Depth")
        ax.invert_yaxis()
        ax_idx += 1

    ax_zone = axes[ax_idx]
    ax_zone.step(zone, depth, where="post", lw=1.5)
    ax_zone.set_xlabel("Predicted zone")
    ax_zone.set_title("Zone backup")
    ax_zone.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax_zone.invert_yaxis()
    ax_idx += 1

    if n_prob_axes:
        ax_prob = axes[ax_idx]
        for pc in prob_cols:
            ax_prob.plot(df_result[pc].values, depth, lw=0.8, label=pc.replace("prob_", ""))
        ax_prob.set_xlim(0, 1)
        ax_prob.set_xlabel("Prob")
        ax_prob.set_title("Zone probs")
        ax_prob.legend(fontsize=6, loc="lower right")
        ax_prob.invert_yaxis()
        ax_idx += 1

    if n_edge_axes:
        ax_edge = axes[ax_idx]
        for ec in edge_cols:
            label = ec.replace("edge_prob_", "")
            ax_edge.plot(df_result[ec].values, depth, lw=0.8, label=label)
        ax_edge.set_xlabel("Edge prob")
        ax_edge.set_title("Marker-specific edge probabilities")
        ax_edge.legend(fontsize=6, loc="lower right")
        ax_edge.invert_yaxis()

    if df_tops is not None and top_cols and well_name in df_tops.index:
        for i, col in enumerate(top_cols):
            td = df_tops.loc[well_name, col]
            if not np.isnan(td):
                for ax in axes:
                    ax.axhline(td, color=colors[i % 10], ls="--", lw=1, alpha=0.8)

    fig.suptitle(f"Well {well_name}", fontsize=12)
    plt.tight_layout()
    if save_fig:
        plt.savefig(save_fig, dpi=150, bbox_inches="tight")
    plt.show()


# =============================================================================
# 10. MAIN PIPELINE
# =============================================================================

def run_pipeline(
    df_log_train: pd.DataFrame,
    df_tops_train: pd.DataFrame,
    training_well_list: List,
    df_log_valid: pd.DataFrame,
    df_tops_valid: pd.DataFrame,
    df_log_test: Optional[pd.DataFrame] = None,
    df_tops_test: Optional[pd.DataFrame] = None,
    top_cols: Optional[List[str]] = None,
    log_cols: List[str] = DEFAULT_LOG_COLS,
    depth_col: str = "DS_INDEX",
    well_col: str = "well_name",
    srate: float = 4.0,
    margin: float = 500.0,
    min_top_fraction: float = 0.65,
    edge_half_width: int = EDGE_HALF_WIDTH,
    edge_sigma_ft: float = EDGE_SIGMA_FT,
    lower_pct: float = 25.0,
    upper_pct: float = 75.0,
    patch_len: int = 16,
    stride: int = 8,
    d_model: int = 128,
    n_heads: int = 8,
    n_layers: int = 3,
    d_ff: int = 256,
    n_markers: int = 0,
    dropout: float = 0.1,
    pe: str = "zeros",
    learn_pe: bool = True,
    rope: bool = False,
    alibi: bool = False,
    revin: bool = False,
    standardize_logs: bool = True,
    epochs: int = 50,
    batch_size: int = 16,
    lr: float = 1e-4,
    weight_decay: float = 1e-4,
    beta: float = 0.99999,
    depth_jitter: int = 0,
    noise_std: float = 0.05,
    top_weight: float = 1.0,
    seg_weight: float = 0.25,
    selection_mode: str = "edge_primary",
    final_eval_mode: str = "edge_primary",
    save_path: str = "patchtst_edge_primary.pt",
    run_decoder_diagnostic_after: bool = True,
):
    if top_cols is None:
        top_cols = DEFAULT_TOP_COLS

    n_classes = len(top_cols) + 1
    n_channels = len(log_cols)

    df_tops_train = df_tops_train.copy()
    df_tops_valid = df_tops_valid.copy()
    df_tops_train.index = df_tops_train.index.astype(str)
    df_tops_valid.index = df_tops_valid.index.astype(str)

    if standardize_logs:
        df_log_train, df_log_valid, df_log_test, log_stats = standardize_logs_train_valid(
            df_log_train,
            df_log_valid,
            log_cols,
            df_test=df_log_test,
        )
    else:
        log_stats = None

    global_min, global_max = compute_global_depth_bounds(
        df_tops_train,
        top_cols,
        margin=margin,
        lower_pct=lower_pct,
        upper_pct=upper_pct,
    )
    print(f"Depth window (pct=[{lower_pct},{upper_pct}]): [{global_min:.1f}, {global_max:.1f}]")

    common_kw = dict(
        top_cols=top_cols,
        global_min=global_min,
        global_max=global_max,
        log_cols=log_cols,
        depth_col=depth_col,
        well_col=well_col,
        srate=srate,
        ignore_index=IGNORE_INDEX,
        min_top_fraction=min_top_fraction,
        edge_half_width=edge_half_width,
        edge_sigma_ft=edge_sigma_ft,
    )

    print("\n--- Training ---")
    X_train, y_train, y_edge_train, _, train_wells = prepare_sequences(
        df_log_train,
        df_tops_train,
        well_list=training_well_list,
        **common_kw,
    )

    print("\n--- Validation ---")
    X_valid, y_valid, y_edge_valid, depth_axes_valid, valid_wells = prepare_sequences(
        df_log_valid,
        df_tops_valid,
        **common_kw,
    )

    seq_len = X_train.shape[2]
    assert X_valid.shape[2] == seq_len, "Sequence length mismatch"

    print(
        f"\nBuilding PatchTSTTopPicker | seq_len={seq_len} | "
        f"n_channels={n_channels} | n_classes={n_classes} | n_tops={len(top_cols)} | "
        f"rope={rope} | alibi={alibi} | pe={pe} | learn_pe={learn_pe}"
    )

    model = PatchTSTTopPicker(
        seq_len=seq_len,
        n_classes=n_classes,
        n_channels=n_channels,
        patch_len=patch_len,
        stride=stride,
        d_model=d_model,
        n_heads=n_heads,
        n_layers=n_layers,
        d_ff=d_ff,
        n_markers=n_markers,
        dropout=dropout,
        head_dropout=dropout,
        pe=pe,
        learn_pe=learn_pe,
        rope=rope,
        alibi=alibi,
        revin=revin,
    )

    model = train_model(
        model=model,
        X_train=X_train,
        y_train=y_train,
        y_edge_train=y_edge_train,
        X_valid=X_valid,
        y_valid=y_valid,
        y_edge_valid=y_edge_valid,
        depth_axes_valid=depth_axes_valid,
        valid_wells=valid_wells,
        df_tops_valid=df_tops_valid,
        top_cols=top_cols,
        log_cols=log_cols,
        epochs=epochs,
        batch_size=batch_size,
        lr=lr,
        weight_decay=weight_decay,
        beta=beta,
        depth_jitter=depth_jitter,
        noise_std=noise_std,
        top_weight=top_weight,
        seg_weight=seg_weight,
        selection_mode=selection_mode,
        save_path=save_path,
        ignore_index=IGNORE_INDEX,
    )

    df_log_inf = df_log_test if df_log_test is not None else df_log_valid
    df_tops_inf = df_tops_test if df_tops_test is not None else df_tops_valid
    df_tops_inf = df_tops_inf.copy()
    df_tops_inf.index = df_tops_inf.index.astype(str)

    print("\n--- Inference ---")
    X_inf, y_inf, y_edge_inf, depth_axes_inf, inf_wells = prepare_sequences(
        df_log_inf,
        df_tops_inf,
        **common_kw,
    )
    assert X_inf.shape[2] == seq_len, "Inference seq_len mismatch"

    print("Running inference...")
    results = predict(
        model,
        X_inf,
        depth_axes_inf,
        log_cols=log_cols,
        top_cols=top_cols,
        batch_size=batch_size,
        zone_decoder="viterbi",
    )

    df_eval = evaluate_top_picking(
        results,
        inf_wells,
        df_tops_inf,
        top_cols,
        tolerance=RECALL_TOLERANCE_FT,
        mode=final_eval_mode,
        verbose=True,
    )

    evaluate_segmentation_metrics(results, inf_wells, df_tops_inf, top_cols)

    diag = None
    if run_decoder_diagnostic_after:
        diag = decoder_diagnostic(
            model=model,
            X=X_inf,
            depth_axes=depth_axes_inf,
            well_names=inf_wells,
            df_tops=df_tops_inf,
            top_cols=top_cols,
            log_cols=log_cols,
            batch_size=batch_size,
        )

    return model, results, df_eval, inf_wells, diag, log_stats


# =============================================================================
# 11. BACKBONE PATCH NOTE FOR ROPE / ALIBI
# =============================================================================

"""
To actually use rope=True or alibi=True, patch PatchTST_backbone.py.

In PatchTST_backbone.__init__, replace:

    rope=False, alibi=False

inside the TSTiEncoder call with:

    rope=rope, alibi=alibi

In TSTiEncoder.__init__, replace:

    rope=False, alibi=False

inside the TSTEncoder call with:

    rope=rope, alibi=alibi

In TSTEncoder.__init__, replace:

    rope=False, alibi=False

inside each TSTEncoderLayer call with:

    rope=rope, alibi=alibi

In TSTEncoderLayer.__init__, replace:

    rope=False, alibi=False

inside _MultiheadAttention with:

    rope=rope, alibi=alibi

Then you can run:

    model, results, df_eval, inf_wells, diag, log_stats = run_pipeline(
        df_log_train=df_log_train,
        df_tops_train=df_tops_train,
        training_well_list=training_well_list,
        df_log_valid=df_log_valid,
        df_tops_valid=df_tops_valid,
        rope=True,
        pe="zeros",
        learn_pe=True,
        selection_mode="edge_primary",
        final_eval_mode="edge_primary",
    )
"""


Using backbone: /var/tmp/seq2seq_v2_backbone_u8py533p/PatchTST_backbone.py


'\nTo actually use rope=True or alibi=True, patch PatchTST_backbone.py.\n\nIn PatchTST_backbone.__init__, replace:\n\n    rope=False, alibi=False\n\ninside the TSTiEncoder call with:\n\n    rope=rope, alibi=alibi\n\nIn TSTiEncoder.__init__, replace:\n\n    rope=False, alibi=False\n\ninside the TSTEncoder call with:\n\n    rope=rope, alibi=alibi\n\nIn TSTEncoder.__init__, replace:\n\n    rope=False, alibi=False\n\ninside each TSTEncoderLayer call with:\n\n    rope=rope, alibi=alibi\n\nIn TSTEncoderLayer.__init__, replace:\n\n    rope=False, alibi=False\n\ninside _MultiheadAttention with:\n\n    rope=rope, alibi=alibi\n\nThen you can run:\n\n    model, results, df_eval, inf_wells, diag, log_stats = run_pipeline(\n        df_log_train=df_log_train,\n        df_tops_train=df_tops_train,\n        training_well_list=training_well_list,\n        df_log_valid=df_log_valid,\n        df_tops_valid=df_tops_valid,\n        rope=True,\n        pe="zeros",\n        learn_pe=True,\n        selection_

In [2]:
# __CARBON_SETUP__
# Carbon + energy tracking for THIS notebook, replicating the pattern used in
# NPD_2_Lithoformer_G2_final_copy_carbon.ipynb (which itself replicates
# Enora/[np]carbon_tracking.ipynb):
#   - CodeCarbon OfflineEmissionsTracker  -> CPU + GPU + RAM energy, NOR-pinned emissions
#   - CarbonTracker                       -> GPU-only energy + live-grid emissions
# Measured for TWO scopes separately (sequential, no overlapping tracker instances):
#   "train"  = run_pipeline call (fit + validation eval)
#   "infer"  = test-set prediction (prepare_sequences + predict + evaluate_top_picking)
# The dump cell sums them into "total" (= train + infer) and writes one flat JSON.
import os, re, time, json, socket, subprocess, uuid
from pathlib import Path
from datetime import datetime
import numpy as _np
import pandas as _pd

from carbontracker.tracker import CarbonTracker
from codecarbon import OfflineEmissionsTracker

# Deterministic CUDA so re-runs reproduce (same as the reference notebook).
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
try:
    import torch as _torch
    _torch.backends.cudnn.deterministic = True
    _torch.backends.cudnn.benchmark = False
    try:
        _torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception as _e:
        print(f"[carbon] could not enable deterministic algorithms: {_e}")
except Exception:
    pass

CARBON_ARCH_LABEL   = "seq2seq_transformer_v2_colorado"
CARBON_COUNTRY_ISO3 = "NOR"
CARBON_LOG_DIR      = Path("carbon_logs_seq2seq_v2_colorado"); CARBON_LOG_DIR.mkdir(exist_ok=True)

# Optional ElectricityMaps key for CarbonTracker live grid (else CT uses its
# built-in averages / detected location).
os.environ.setdefault("ELECTRICITYMAPS_API_KEY", "")
_CARBON_CT_KEYS = {"electricitymaps": os.environ["ELECTRICITYMAPS_API_KEY"]} if os.environ.get("ELECTRICITYMAPS_API_KEY") else None

# Per-scope state: scope -> {tracker handles, timing, parsed numbers}.
_CARBON_SCOPES = {}


def _carbon_start(scope):
    """Start CodeCarbon + CarbonTracker for one scope. Clears stale logs for
    this scope first so the parsers can only see the current run."""
    run_name = f"{CARBON_ARCH_LABEL}_{scope}"
    for p in CARBON_LOG_DIR.glob(f"{run_name}_*"):
        try: p.unlink()
        except Exception: pass
    cc_csv = CARBON_LOG_DIR / f"{run_name}_codecarbon.csv"
    if cc_csv.exists():
        try: cc_csv.unlink()
        except Exception: pass

    cc = OfflineEmissionsTracker(
        country_iso_code=CARBON_COUNTRY_ISO3,
        project_name=run_name,
        output_dir=str(CARBON_LOG_DIR),
        output_file=f"{run_name}_codecarbon.csv",
        measure_power_secs=5, tracking_mode="machine",
        log_level="error", save_to_file=True,
    )
    ct = CarbonTracker(
        epochs=1, monitor_epochs=1, epochs_before_pred=1,
        log_dir=str(CARBON_LOG_DIR), log_file_prefix=run_name + "_",
        api_keys=_CARBON_CT_KEYS, verbose=1,
    )
    cc.start()
    try: ct.epoch_start()
    except Exception: pass
    _CARBON_SCOPES[scope] = {"cc": cc, "ct": ct, "run_name": run_name,
                             "t0": time.time(), "elapsed_s": float("nan")}
    print(f"[carbon] started scope={scope!r} (CodeCarbon NOR + CarbonTracker)")


def _carbon_stop(scope):
    """Stop both trackers for a scope and record elapsed wall time."""
    st = _CARBON_SCOPES.get(scope)
    if not st:
        print(f"[carbon] WARN: stop called for unknown scope {scope!r}")
        return
    try: st["ct"].epoch_end(); st["ct"].stop()
    except Exception: pass
    try: st["cc"].stop()
    except Exception: pass
    st["elapsed_s"] = time.time() - st["t0"]
    print(f"[carbon] stopped scope={scope!r}  elapsed={st['elapsed_s']:.1f}s")


print(f"[carbon] setup OK -> arch={CARBON_ARCH_LABEL}, logs in {CARBON_LOG_DIR}/")


[carbon] setup OK -> arch=seq2seq_transformer_v2_colorado, logs in carbon_logs_seq2seq_v2_colorado/


In [3]:
TOP_COLS   = ["NIOBARA", "CODELL", "FORT_HAYS"]   # physical (increasing-depth) order
LOG_COLS   = ["GR_CLEAN"]                          # Colorado has a single GR channel
DEPTH_COL  = "DS_INDEX"
WELL_COL   = "well_name"

_CD = "data"
df_log_train  = pd.read_parquet(f"{_CD}/logs_train.parquet")
df_log_valid  = pd.read_parquet(f"{_CD}/logs_valid.parquet")
df_log_test   = pd.read_parquet(f"{_CD}/logs_test.parquet")
df_tops_train = pd.read_parquet(f"{_CD}/tops_train.parquet")
df_tops_valid = pd.read_parquet(f"{_CD}/tops_valid.parquet")
df_tops_test  = pd.read_parquet(f"{_CD}/tops_test.parquet")

for df in [df_tops_train, df_tops_valid, df_tops_test]:
    if WELL_COL in df.columns:
        df.set_index(WELL_COL, inplace=True)

with open(f"{_CD}/train_wells.json") as f:
    training_well_list = json.load(f)

print(f"Loaded: train={len(training_well_list)} wells, "
      f"valid tops={len(df_tops_valid)}, test tops={len(df_tops_test)}")


Loaded: train=540 wells, valid tops=60, test tops=100


In [4]:
import time as _time  # __CARBON_TIMER__
_t0 = _time.time()
reset_seeds(42)
_carbon_start("train")  # measure TRAIN scope (run_pipeline = fit + valid eval)
try:
    model, results, df_eval, inf_wells, diag, log_stats = run_pipeline(
        df_log_train=df_log_train,
        df_tops_train=df_tops_train,
        training_well_list=training_well_list,
        df_log_valid=df_log_valid,
        df_tops_valid=df_tops_valid,
        df_log_test=df_log_test,
        df_tops_test=df_tops_test,
        top_cols=TOP_COLS,
        log_cols=LOG_COLS,
        depth_col=DEPTH_COL,
        well_col=WELL_COL,

        srate=4.0,
        margin=750.0,
        min_top_fraction=0.65,
        edge_half_width=30,

        lower_pct=5.0,
        upper_pct=95.0,

        patch_len=16,
        stride=8,
        d_model=128,
        n_heads=8,
        n_layers=3,
        d_ff=256,
        n_markers=0,
        dropout=0.1,

        epochs=30,
        batch_size=16,
        lr=1e-4,
        weight_decay=1e-4,
        beta=0.99999,

        # edge/top localization is now controlled by top_weight
        top_weight=0.8,

        # segmentation is backup
        seg_weight=0.25,

        depth_jitter=0,
        noise_std=0.00,

        save_path="patchtst_edge_primary.pt",
        run_decoder_diagnostic_after=False,
    )
finally:
    _carbon_stop("train")

_train_elapsed_s = _time.time() - _t0
print(f'[carbon timer] training elapsed: {_train_elapsed_s:.1f}s')


[codecarbon INFO @ 22:13:10] offline tracker init
[codecarbon WARNING @ 22:13:10] Multiple instances of codecarbon are allowed to run at the same time.


CarbonTracker: INFO - Detected CPU: Intel(R) Xeon(R) CPU @ 2.00GHz
CarbonTracker: WARNING - No API keys provided. Skipping intensity provider initialization.
CarbonTracker: WARNING - No matching TDP found for CPU: Intel(R) Xeon(R) CPU @ 2.00GHz. Using average TDP of 35.61W at 50% utilization as fallback.
CarbonTracker: The following components were found: GPU with device(s) Tesla T4. CPU with device(s) Intel(R) Xeon(R) CPU @ 2.00GHz.
CarbonTracker: WARNING - No carbon intensity provider specified. Using average carbon intensity for NL: 253.31 gCO2eq/kWh.
[carbon] started scope='train' (CodeCarbon NOR + CarbonTracker)
Depth window (pct=[5.0,95.0]): [4000.0, 8000.0]

--- Training ---
Prepared 538 wells | seq_len=1001 | n_channels=1 | n_classes=4 | global range=[4000.0, 8000.0]

--- Validation ---
Prepared 59 wells | seq_len=1001 | n_channels=1 | n_classes=4 | global range=[4000.0, 8000.0]

Building PatchTSTTopPicker | seq_len=1001 | n_channels=1 | n_classes=4 | n_tops=3 | rope=False | al

/home/jupyter/CRP-autoresearch/.venv/lib/python3.10/site-packages/torch/nn/functional.py:3104: UserWarning: nll_loss2d_forward_out_cuda_template does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:83.)
  return torch._C._nn.cross_entropy_loss(input, target, weight, _Reduction.get_enum(reduction), ignore_index, label_smoothing)
/home/jupyter/CRP-autoresearch/.venv/lib/python3.10/site-packages/torch/autograd/graph.py:769: UserWarning: upsample_linear1d_backward_out_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered in

Epoch   1/30 | train_loss=5.0888 | train_top=5.9675 | train_seg=1.2593 | val_top=5.3777 | val_seg=1.2972 | val_seg_acc=0.5100 | val_macro_recall@50=0.5924 | mode=edge_primary
  -> Saved best checkpoint: patchtst_edge_primary.pt | val_macro_recall@50=0.5924


/home/jupyter/CRP-autoresearch/.venv/lib/python3.10/site-packages/torch/nn/functional.py:3104: UserWarning: nll_loss2d_forward_out_cuda_template does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:83.)
  return torch._C._nn.cross_entropy_loss(input, target, weight, _Reduction.get_enum(reduction), ignore_index, label_smoothing)
/home/jupyter/CRP-autoresearch/.venv/lib/python3.10/site-packages/torch/autograd/graph.py:769: UserWarning: upsample_linear1d_backward_out_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered in

Epoch   2/30 | train_loss=3.9985 | train_top=4.6549 | train_seg=1.0984 | val_top=4.4452 | val_seg=1.1440 | val_seg_acc=0.7446 | val_macro_recall@50=0.6254 | mode=edge_primary
  -> Saved best checkpoint: patchtst_edge_primary.pt | val_macro_recall@50=0.6254


... (52 outputs left out)

/home/jupyter/CRP-autoresearch/.venv/lib/python3.10/site-packages/torch/nn/functional.py:3104: UserWarning: nll_loss2d_forward_out_cuda_template does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:83.)
  return torch._C._nn.cross_entropy_loss(input, target, weight, _Reduction.get_enum(reduction), ignore_index, label_smoothing)
/home/jupyter/CRP-autoresearch/.venv/lib/python3.10/site-packages/torch/autograd/graph.py:769: UserWarning: upsample_linear1d_backward_out_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered in

Epoch  29/30 | train_loss=2.8308 | train_top=3.4220 | train_seg=0.3726 | val_top=3.8710 | val_seg=0.2018 | val_seg_acc=0.9315 | val_macro_recall@50=0.8171 | mode=edge_primary


/home/jupyter/CRP-autoresearch/.venv/lib/python3.10/site-packages/torch/nn/functional.py:3104: UserWarning: nll_loss2d_forward_out_cuda_template does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:83.)
  return torch._C._nn.cross_entropy_loss(input, target, weight, _Reduction.get_enum(reduction), ignore_index, label_smoothing)
/home/jupyter/CRP-autoresearch/.venv/lib/python3.10/site-packages/torch/autograd/graph.py:769: UserWarning: upsample_linear1d_backward_out_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered in

Epoch  30/30 | train_loss=2.8453 | train_top=3.4396 | train_seg=0.3744 | val_top=3.8763 | val_seg=0.2039 | val_seg_acc=0.9314 | val_macro_recall@50=0.8171 | mode=edge_primary

--- Inference ---
Prepared 100 wells | seq_len=1001 | n_channels=1 | n_classes=4 | global range=[4000.0, 8000.0]
Running inference...

Top-picking errors and recall@50ft | mode=edge_primary
      top   MAE  Median  Recall@50ft  TP   n
  NIOBARA 11.71     8.0         0.97  97 100
   CODELL  9.62     8.0         1.00 100 100
FORT_HAYS  6.25     5.0         1.00 100 100

Macro-recall@50ft (PRIMARY): 0.9900
Pooled-recall@50ft         : 0.9900

Overall depth-sample accuracy : 0.9769  (on 73,790 labelled points)
Macro-average recall          : 0.9621
Per-class recall              : zone_0=0.985  zone_1=0.912  zone_2=0.993  zone_3=0.958

Per-class metrics:
              precision    recall  f1-score   support

      zone_0       1.00      0.98      0.99     61498
      zone_1       0.86      0.91      0.88      6156
   

In [5]:
df_log_test_scaled = df_log_test.copy()

for col in LOG_COLS:
    mu, sd = log_stats[col]
    df_log_test_scaled[col] = (df_log_test_scaled[col] - mu) / sd

In [6]:
# Recompute the depth bounds the way cell 2's run_pipeline does internally,
# so this cell can be executed top-to-bottom without
# depending on globals that only live inside run_pipeline.
import time as _time  # __CARBON_INFER_TIMER__
_t0_infer = _time.time()
_carbon_start("infer")  # measure INFERENCE scope (test-set prediction)
try:
    global_min, global_max = compute_global_depth_bounds(
        df_tops_train, TOP_COLS, margin=750.0, lower_pct=5.0, upper_pct=95.0,
    )
    print(f'[test-eval] global_min={global_min:.1f}  global_max={global_max:.1f}')

    X_test, y_test, y_edge_test, depth_axes_test, test_wells_used = prepare_sequences(
        df_log_test_scaled,
        df_tops_test,
        top_cols=TOP_COLS,
        global_min=global_min,
        global_max=global_max,
        log_cols=LOG_COLS,
        depth_col=DEPTH_COL,
        well_col=WELL_COL,
        srate=4.0,
        min_top_fraction=0.65,
        edge_half_width=30,
    )

    results_test = predict(
        model,
        X_test,
        depth_axes_test,
        log_cols=LOG_COLS,
        top_cols=TOP_COLS,
        batch_size=16,
        zone_decoder="viterbi",
    )

    df_eval_test = evaluate_top_picking(
        results_test,
        test_wells_used,
        df_tops_test,
        TOP_COLS,
        tolerance=RECALL_TOLERANCE_FT,
        mode="edge_windowed",
        window_ft=800.0,
        verbose=True,
    )

    evaluate_segmentation_metrics(
        results_test,
        test_wells_used,
        df_tops_test,
        TOP_COLS,
    )
finally:
    _carbon_stop("infer")
_infer_elapsed_s = _time.time() - _t0_infer
print(f'[carbon timer] inference elapsed: {_infer_elapsed_s:.1f}s')


CarbonTracker: INFO - Detected CPU: Intel(R) Xeon(R) CPU @ 2.00GHz
CarbonTracker: WARNING - No API keys provided. Skipping intensity provider initialization.
CarbonTracker: WARNING - No matching TDP found for CPU: Intel(R) Xeon(R) CPU @ 2.00GHz. Using average TDP of 35.61W at 50% utilization as fallback.
CarbonTracker: The following components were found: GPU with device(s) Tesla T4. CPU with device(s) Intel(R) Xeon(R) CPU @ 2.00GHz.
CarbonTracker: WARNING - No carbon intensity provider specified. Using average carbon intensity for NL: 253.31 gCO2eq/kWh.
[carbon] started scope='infer' (CodeCarbon NOR + CarbonTracker)
[test-eval] global_min=4000.0  global_max=8000.0
Prepared 100 wells | seq_len=1001 | n_channels=1 | n_classes=4 | global range=[4000.0, 8000.0]

Top-picking errors and recall@50ft | mode=edge_windowed
      top   MAE  Median  Recall@50ft  TP   n
  NIOBARA 11.71     8.0         0.97  97 100
   CODELL  9.62     8.0         1.00 100 100
FORT_HAYS  6.25     5.0         1.00 10

In [7]:
# __CARBON_METRICS_DUMP__
# Self-contained metrics + carbon dump (one flat JSON per run). Mirrors the
# pattern in NPD_2_Lithoformer_G2_final_copy_carbon.ipynb so the downstream
# carbon-comparison notebook can read every model's JSON identically.
#
# Scopes measured by _carbon_start/_carbon_stop (see __CARBON_SETUP__ cell):
#   train_*  = run_pipeline (fit + validation eval)
#   infer_*  = test-set prediction (prepare_sequences + predict + evaluate_top_picking)
#   total_*  = train + infer  (energy/emissions are additive)
# Each scope carries BOTH libraries:
#   *_ct_*  = CarbonTracker  (GPU-only energy + live-grid emissions)
#   *_cc_*  = CodeCarbon      (CPU+GPU+RAM energy, NOR-pinned emissions)
import os, re, json
import numpy as np
import pandas as pd

_out_path = os.environ.get("CARBON_METRICS_OUT", "metrics_seq2seq_v2_colorado.json")


def _parse_carbontracker_log(log_dir, prefix):
    """Robust CT parser, identical to Enora/[np]carbon_tracking.ipynb."""
    from pathlib import Path
    files = sorted(list(Path(log_dir).glob(f"{prefix}*carbontracker*.log")),
                   key=lambda p: p.stat().st_mtime, reverse=True)
    if not files:
        return float("nan"), float("nan")
    for path in files:
        text = path.read_text()
        for heading in ("Actual consumption", "Predicted consumption", "Final emissions", "Total"):
            m = re.search(heading + r".*?Energy:\s*([\d.eE+-]+)\s*kWh.*?(?:CO2eq|CO2):\s*([\d.eE+-]+)\s*g",
                          text, re.DOTALL | re.IGNORECASE)
            if m:
                return float(m.group(1)), float(m.group(2))
    text = files[0].read_text()
    kwh_m = re.findall(r"([\d.eE+-]+)\s*kWh", text)
    co2_m = re.findall(r"([\d.eE+-]+)\s*g\s*(?:CO2eq|CO2|of\s*CO2)", text, re.IGNORECASE)
    if kwh_m and co2_m:
        return float(kwh_m[-1]), float(co2_m[-1])
    return float("nan"), float("nan")


def _read_codecarbon_csv(path):
    from pathlib import Path
    if not Path(path).exists():
        return {"emissions_kg": float("nan"), "energy_kwh": float("nan"),
                "cpu_kwh": float("nan"), "gpu_kwh": float("nan"), "ram_kwh": float("nan"),
                "country": None, "region": None}
    r = pd.read_csv(path).iloc[-1]
    return {"emissions_kg": float(r.get("emissions", float("nan"))),
            "energy_kwh": float(r.get("energy_consumed", float("nan"))),
            "cpu_kwh": float(r.get("cpu_energy", float("nan"))),
            "gpu_kwh": float(r.get("gpu_energy", float("nan"))),
            "ram_kwh": float(r.get("ram_energy", float("nan"))),
            "country": r.get("country_name", None), "region": r.get("region", None)}


def _scope_fields(scope):
    run_name = f"{CARBON_ARCH_LABEL}_{scope}"
    ct_kwh, ct_g = _parse_carbontracker_log(CARBON_LOG_DIR, prefix=run_name + "_")
    cc = _read_codecarbon_csv(CARBON_LOG_DIR / f"{run_name}_codecarbon.csv")
    cc_kg = cc["emissions_kg"]
    st = _CARBON_SCOPES.get(scope, {})
    return {
        f"{scope}_elapsed_s":       float(st.get("elapsed_s", float("nan"))),
        f"{scope}_ct_energy_kwh":   ct_kwh,
        f"{scope}_ct_emissions_g":  ct_g,
        f"{scope}_cc_emissions_kg": cc_kg,
        f"{scope}_cc_emissions_g":  cc_kg * 1000 if not np.isnan(cc_kg) else float("nan"),
        f"{scope}_cc_energy_kwh":   cc["energy_kwh"],
        f"{scope}_cc_cpu_kwh":      cc["cpu_kwh"],
        f"{scope}_cc_gpu_kwh":      cc["gpu_kwh"],
        f"{scope}_cc_ram_kwh":      cc["ram_kwh"],
        f"{scope}_cc_country":      cc["country"],
        f"{scope}_cc_region":       cc["region"],
    }


_train = _scope_fields("train")
_infer = _scope_fields("infer")

# total = train + infer (nansum: NaN+x=x, NaN+NaN=NaN). Country = whichever is set.
_ADD = ["ct_energy_kwh", "ct_emissions_g", "cc_emissions_kg", "cc_emissions_g",
        "cc_energy_kwh", "cc_cpu_kwh", "cc_gpu_kwh", "cc_ram_kwh", "elapsed_s"]
_total = {}
for _k in _ADD:
    _a, _b = _train.get(f"train_{_k}", float("nan")), _infer.get(f"infer_{_k}", float("nan"))
    _total[f"total_{_k}"] = float("nan") if (np.isnan(_a) and np.isnan(_b)) else float(np.nansum([_a, _b]))
_total["total_cc_country"] = _train.get("train_cc_country") or _infer.get("infer_cc_country")


# ── Model metrics (separate from carbon) ────────────────────────────────────
def _count_params(m):
    try:
        return int(sum(p.numel() for p in m.parameters() if p.requires_grad))
    except Exception:
        return None


def _macro_recall(df_eval_, tops):
    """Recompute macro_recall@50 from the err columns."""
    recs = []
    for c in tops:
        err = df_eval_[f"{c}_err"].dropna().values.astype(float)
        if len(err) == 0:
            continue
        recs.append(float((err <= 50.0).mean()))
    return float(np.mean(recs)) if recs else float("nan")


def _mae(df_eval_, tops):
    vals = []
    for c in tops:
        vals.extend(df_eval_[f"{c}_err"].dropna().tolist())
    return float(np.mean(vals)) if vals else float("nan")


_test_recall = _macro_recall(df_eval_test, TOP_COLS) if "df_eval_test" in dir() else float("nan")
_test_mae    = _mae(df_eval_test, TOP_COLS)         if "df_eval_test" in dir() else float("nan")
_test_n      = int(len(df_eval_test))               if "df_eval_test" in dir() else 0
_val_recall  = _macro_recall(df_eval, TOP_COLS)     if "df_eval" in dir() else float("nan")
_val_n       = int(len(df_eval))                    if "df_eval" in dir() else 0

try:
    _params = _count_params(model)
except Exception:
    _params = None

_payload = {
    "arch_label": CARBON_ARCH_LABEL if "CARBON_ARCH_LABEL" in dir() else "seq2seq_transformer_v2_colorado",
    "country_iso": CARBON_COUNTRY_ISO3 if "CARBON_COUNTRY_ISO3" in dir() else "NOR",
    "metric_tolerance_ft": 50.0,
    # model metrics
    "test_macro_recall_at_tol": _test_recall,
    "test_mae_ft": _test_mae,
    "test_n_wells": _test_n,
    "valid_macro_recall_at_tol": _val_recall,
    "valid_n_wells": _val_n,
    "total_params": _params,
    "train_elapsed_s": float(_train_elapsed_s) if "_train_elapsed_s" in dir() else _train.get("train_elapsed_s", float("nan")),
    "infer_elapsed_s": float(_infer_elapsed_s) if "_infer_elapsed_s" in dir() else _infer.get("infer_elapsed_s", float("nan")),
    "epochs_configured": 30,
    "batch_size": 16,
    "patch_len": 16,
    "stride": 8,
    "d_model": 128,
    "n_layers": 3,
    "seq_len": int(model.seg_head.seq_len) if ("model" in dir() and hasattr(model, "seg_head")) else None,
}
_payload.update(_train)
_payload.update(_infer)
_payload.update(_total)

# Quick grid-intensity sanity print (Norway band 5-80 g/kWh) on the total.
_tot_kwh = _payload.get("total_cc_energy_kwh", float("nan"))
_tot_g   = _payload.get("total_cc_emissions_g", float("nan"))
_implied = (_tot_g / _tot_kwh) if (_tot_kwh and not np.isnan(_tot_kwh) and _tot_kwh > 0) else float("nan")
print(f"[carbon] TOTAL (train+infer): CC energy={_tot_kwh*1000 if not np.isnan(_tot_kwh) else float('nan'):.2f} Wh  "
      f"CC emissions={_tot_g:.3f} g  -> implied {_implied:.1f} g/kWh")
print(f"[carbon] test_recall@50={_test_recall:.3f}  valid_recall={_val_recall:.3f}  params={_params}")

with open(_out_path, "w") as _f:
    json.dump(_payload, _f, indent=2)
print(f"[carbon metrics] wrote {_out_path}  ({len(_payload)} keys)")


[carbon] TOTAL (train+infer): CC energy=2.07 Wh  CC emissions=0.062 g  -> implied 30.1 g/kWh
[carbon] test_recall@50=0.990  valid_recall=0.990  params=516233
[carbon metrics] wrote metrics_seq2seq_v2_colorado.json  (48 keys)


In [8]:
# --- Decoder diagnostic (run standalone; NO retrain required) ---
# Sweeps the edge/viterbi/edge_windowed decoders on the ALREADY-TRAINED `model`
# over the Lithoformer-aligned test set, reusing the test sequences built in the
# test-eval cell above (X_test, depth_axes_test, test_wells_used). Equivalent to
# run_pipeline(run_decoder_diagnostic_after=True) but on the test set and without
# re-fitting. Just execute this one cell after a full run.
diag_test = decoder_diagnostic(
    model=model,
    X=X_test,
    depth_axes=depth_axes_test,
    well_names=test_wells_used,
    df_tops=df_tops_test,
    top_cols=TOP_COLS,
    log_cols=LOG_COLS,
    batch_size=16,
)
diag_test



Decoder diagnostic summary:
      decoder  param  macro_recall_at_tol  pooled_recall_at_tol  tolerance_ft
edge_windowed  100.0             0.993333              0.993333          50.0
 edge_primary    0.0             0.990000              0.990000          50.0
edge_windowed  200.0             0.990000              0.990000          50.0
edge_windowed  500.0             0.990000              0.990000          50.0
edge_windowed  300.0             0.990000              0.990000          50.0
edge_windowed  800.0             0.990000              0.990000          50.0
      viterbi    0.0             0.906667              0.906667          50.0


,decoder,param,macro_recall_at_tol,pooled_recall_at_tol,tolerance_ft
2,edge_windowed,100.0,0.993333,0.993333,50.0
0,edge_primary,0.0,0.990000,0.990000,50.0
3,edge_windowed,200.0,0.990000,0.990000,50.0
5,edge_windowed,500.0,0.990000,0.990000,50.0
4,edge_windowed,300.0,0.990000,0.990000,50.0
6,edge_windowed,800.0,0.990000,0.990000,50.0
1,viterbi,0.0,0.906667,0.906667,50.0
